In [1]:
import torch
import pandas as pd
from torch import nn
from torch import optim
from torch.utils.data import Dataset, DataLoader


class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x = df.iloc[:, 0].values
        self.y = df.iloc[:, 1].values
        self.length = len(df)

    def __getitem__(self, index):
        x = torch.FloatTensor([self.x[index] ** 2, self.x[index]])
        y = torch.FloatTensor([self.y[index]])
        return x, y

    def __len__(self):
        return self.length

class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(2, 1)

    def forward(self, x):
        x = self.layer(x)
        return x

In [2]:
train_dataset = CustomDataset("../datasets/non_linear.csv")
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last=True)

In [3]:

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CustomModel().to(device)
criterion = nn.MSELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.0001)

In [4]:

checkpoint = torch.load("../models/checkpoint-6.pt")
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
checkpoint_epoch = checkpoint["epoch"]
checkpoint_description = checkpoint["description"]
print(checkpoint_description)

CustomModel 체크포인트-6


In [5]:

for epoch in range(checkpoint_epoch + 1, 10000):
    cost = 0.0

    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        loss = criterion(output, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss
        if (epoch + 1) % 1000 == 0:
            print(f"Epoch : {epoch+1:4d}, Model : {list(model.parameters())}, Cost : {cost:.3f}")

Epoch : 7000, Model : [Parameter containing:
tensor([[ 3.1083, -1.7029]], device='cuda:0', requires_grad=True), Parameter containing:
tensor([-0.0127], device='cuda:0', requires_grad=True)], Cost : 0.192
Epoch : 8000, Model : [Parameter containing:
tensor([[ 3.1075, -1.7026]], device='cuda:0', requires_grad=True), Parameter containing:
tensor([0.0305], device='cuda:0', requires_grad=True)], Cost : 0.194
Epoch : 9000, Model : [Parameter containing:
tensor([[ 3.1070, -1.7028]], device='cuda:0', requires_grad=True), Parameter containing:
tensor([0.0700], device='cuda:0', requires_grad=True)], Cost : 0.155
Epoch : 10000, Model : [Parameter containing:
tensor([[ 3.1068, -1.7029]], device='cuda:0', requires_grad=True), Parameter containing:
tensor([0.1060], device='cuda:0', requires_grad=True)], Cost : 0.123
